# SYNTHIA -> Cityscapes Final Resource Pipeline (1000 Images)

This notebook is the final low-resource run:

| Step | Description |
|------|-------------|
| 0 | Mount Drive, clone repo, install deps, set paths |
| 1 | Prepare SYNTHIA and Cityscapes data |
| 2 | Build multilabel JSONs for DAMP |
| 3 | Train/load the existing DAMP checkpoint |
| 4 | Generate zero-shot and DAMP prompt-only CAMs for 1000 SYNTHIA images |
| 5 | Build a class-wise hybrid CAM set |
| 6 | Evaluate zero-shot, prompt-only, and hybrid on the same 1000-image split |
| 7 | CRF the selected best CAM set into pseudo masks |
| 8 | Export image/mask pairs for segmentation training |

Default best source is `hybrid`, but after Step 6 you can change `BEST_CAM_KIND` to `zero` or `prompt` before Step 7 if the metrics say otherwise.


In [ ]:
# ===== CELL 0a: MOUNT DRIVE =====
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== CELL 0b: CLONE / UPDATE REPO + INSTALL DEPS =====
import os, sys, subprocess

REPO_DIR = '/content/Damp_es'
REPO_URL = 'https://github.com/baominh5xx2/Damp_es_CS338.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Repo already exists at {REPO_DIR}; updating with git pull --ff-only')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=False)
    pull = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=False)
    if pull.returncode != 0:
        print('WARNING: git pull failed, likely because the Colab repo has local edits.')
        print('If you need a clean update, run: !rm -rf /content/Damp_es and rerun this cell.')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=False)

# Install dependencies.
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q timm yacs ftfy regex pydensecrf lxml tqdm huggingface_hub


In [ ]:
# ===== CELL 0c: CONFIG - paths & settings =====
from pathlib import Path

# Change this to your Drive path if needed.
DATA_ROOT = Path('/content/drive/MyDrive/datasets/synthia_cs338')
OUTPUT_DIR = DATA_ROOT / 'output'

# Final low-resource run.
RUN_NAME = 'synthia_clipnorm_tau052_e3'
CAM_MAX_IMAGES = 1000
EVAL_MAX_IMAGES = CAM_MAX_IMAGES
BEST_CAM_KIND = 'hybrid'  # options: 'zero', 'prompt', 'hybrid'
CRF_CONFIDENCE = 0.95
CRF_N_JOBS = 1
USE_CRF = False  # final-resource default: False is much faster
PSEUDO_MASK_THRESHOLD = 0.10  # set this to the best threshold from Step 7

# Derived paths.
SYNTHIA_RAW = DATA_ROOT / 'data' / 'raw' / 'synthia'
CITY_RAW = DATA_ROOT / 'data' / 'raw' / 'cityscapes'
PROCESSED = DATA_ROOT / 'data' / 'processed'

DAMP_DIR = OUTPUT_DIR / 'damp' / RUN_NAME
CAM_ZERO_DIR = OUTPUT_DIR / 'synthia' / f'cams_zero_raw_{CAM_MAX_IMAGES}'
CAM_PROMPT_DIR = OUTPUT_DIR / 'synthia' / f'cams_damp_{RUN_NAME}_prompt_only_raw_{CAM_MAX_IMAGES}'
CAM_HYBRID_DIR = OUTPUT_DIR / 'synthia' / f'cams_hybrid_zero_prompt_{RUN_NAME}_{CAM_MAX_IMAGES}'
CAM_DIR_BY_KIND = {
    'zero': CAM_ZERO_DIR,
    'prompt': CAM_PROMPT_DIR,
    'hybrid': CAM_HYBRID_DIR,
}
BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]

MASK_DIR = OUTPUT_DIR / 'synthia' / f'pseudo_masks_{BEST_CAM_KIND}_{RUN_NAME}_{CAM_MAX_IMAGES}'
SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
SEG_TRAIN_PAIRS = SEG_EXPORT_DIR / f'train_pairs_{BEST_CAM_KIND}_first{CAM_MAX_IMAGES}.txt'

SYNTHIA_IMG = SYNTHIA_RAW / 'images'
SYNTHIA_LBL = SYNTHIA_RAW / 'labels'
SYNTHIA_SPLIT = SYNTHIA_RAW / 'splits' / 'train.txt'
SYNTHIA_CAM_SPLIT = SYNTHIA_RAW / 'splits' / f'train_first{CAM_MAX_IMAGES}.txt'

CITY_IMG = CITY_RAW / 'images'
CITY_LBL = CITY_RAW / 'labels'
CITY_TRAIN_SPLIT = CITY_RAW / 'splits' / 'train.txt'
CITY_VAL_SPLIT = CITY_RAW / 'splits' / 'val.txt'

HF_SYNTHIA_REPO = 'Minhbao5xx2/synthia-rand-cityscapes-16class-parquet_fix'
HF_CITY_REPO = 'Chris1/cityscapes'

print(f'DATA_ROOT        : {DATA_ROOT}')
print(f'RUN_NAME         : {RUN_NAME}')
print(f'CAM_MAX_IMAGES   : {CAM_MAX_IMAGES}')
print(f'DAMP_DIR         : {DAMP_DIR}')
print(f'CAM_ZERO_DIR     : {CAM_ZERO_DIR}')
print(f'CAM_PROMPT_DIR   : {CAM_PROMPT_DIR}')
print(f'CAM_HYBRID_DIR   : {CAM_HYBRID_DIR}')
print(f'BEST_CAM_KIND    : {BEST_CAM_KIND}')
print(f'BEST_CAM_DIR     : {BEST_CAM_DIR}')
print(f'MASK_DIR         : {MASK_DIR}')
print(f'SEG_TRAIN_PAIRS  : {SEG_TRAIN_PAIRS}')


## Step 1: Download & Prepare SYNTHIA

Downloads from HuggingFace (fixed parquet with correct 16-bit labels), then converts to images/labels/splits.

In [ ]:
# ===== CELL 1: DOWNLOAD + PREPARE SYNTHIA =====
import os, glob
from pathlib import Path

PARQUET_DIR = DATA_ROOT / "synthia_parquet"

# ── 1a: Download parquet from HuggingFace ──
if not SYNTHIA_SPLIT.exists():
    print("Downloading SYNTHIA parquet from HuggingFace ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_SYNTHIA_REPO,
        repo_type="dataset",
        local_dir=str(PARQUET_DIR),
        max_workers=8,
    )
    print("Download complete.")
else:
    print("SYNTHIA data already prepared, skipping download.")

# ── 1b: Convert parquet → images/labels/splits ──
if not SYNTHIA_SPLIT.exists():
    print("Converting parquet to images/labels/splits ...")
    !python {REPO_DIR}/tools/prepare_synthia_hf.py \
        --parquet-dir {PARQUET_DIR / "parquet"} \
        --output-root {SYNTHIA_RAW} \
        --num-workers 4
else:
    print("SYNTHIA splits already exist, skipping conversion.")

# ── 1c: Verify ──
n_img = len(list(Path(SYNTHIA_IMG).glob("*.png"))) if SYNTHIA_IMG.exists() else 0
n_lbl = len(list(Path(SYNTHIA_LBL).glob("*.png"))) if SYNTHIA_LBL.exists() else 0
n_split = len(open(SYNTHIA_SPLIT).readlines()) if SYNTHIA_SPLIT.exists() else 0
print(f"\nSYNTHIA: {n_img} images, {n_lbl} labels, {n_split} split entries")

# Quick label sanity check
if n_lbl > 0:
    import cv2, numpy as np
    sample_lbl = sorted(Path(SYNTHIA_LBL).glob("*.png"))[0]
    arr = cv2.imread(str(sample_lbl), cv2.IMREAD_UNCHANGED)
    if arr.ndim == 3:
        unique = np.unique(arr[:,:,2])  # Red channel = class ID (16-bit)
    else:
        unique = np.unique(arr)
    n_classes = len([v for v in unique if v != 0])
    print(f"Label check ({sample_lbl.name}): {n_classes} valid classes, unique IDs: {unique.tolist()[:15]}")

## Step 2: Download & Prepare Cityscapes

In [ ]:
# ===== CELL 2: DOWNLOAD + PREPARE CITYSCAPES =====
if not CITY_VAL_SPLIT.exists():
    print("Downloading & preparing Cityscapes from HuggingFace ...")
    !python {REPO_DIR}/tools/prepare_cityscapes_hf.py \
        --dataset-id {HF_CITY_REPO} \
        --output-root {CITY_RAW} \
        --splits train,validation \
        --num-workers 8
else:
    print("Cityscapes already prepared.")

# Verify
n_city_img = len(list(Path(CITY_IMG).glob("*.png"))) if CITY_IMG.exists() else 0
n_city_train = len(open(CITY_TRAIN_SPLIT).readlines()) if CITY_TRAIN_SPLIT.exists() else 0
n_city_val = len(open(CITY_VAL_SPLIT).readlines()) if CITY_VAL_SPLIT.exists() else 0
print(f"Cityscapes: {n_city_img} images, {n_city_train} train, {n_city_val} val")

## Step 3: Build Multilabel JSON

Extracts per-image class labels from segmentation masks for multi-label classification.

In [ ]:
# ===== CELL 3: BUILD MULTILABEL JSON =====
SYNTHIA_ML_DIR = PROCESSED / "synthia_multilabel"
CITY_ML_DIR    = PROCESSED / "cityscapes_multilabel"

# ── SYNTHIA multilabel ──
synthia_ml_file = SYNTHIA_ML_DIR / "multilabel.json"
if not synthia_ml_file.exists():
    print("Building SYNTHIA multilabel ...")
    !python {REPO_DIR}/tools/build_synthia_multilabel.py \
        --split-file {SYNTHIA_SPLIT} \
        --label-dir {SYNTHIA_LBL} \
        --output-dir {SYNTHIA_ML_DIR} \
        --num-workers 8
else:
    print("SYNTHIA multilabel already exists.")

# ── Cityscapes train multilabel ──
city_train_ml = CITY_ML_DIR / "train_multilabel.json"
if not city_train_ml.exists():
    print("Building Cityscapes train multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_TRAIN_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file train_multilabel.json \
        --num-workers 8
else:
    print("Cityscapes train multilabel already exists.")

# ── Cityscapes val multilabel ──
city_val_ml = CITY_ML_DIR / "val_multilabel.json"
if not city_val_ml.exists():
    print("Building Cityscapes val multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_VAL_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file val_multilabel.json \
        --num-workers 8
else:
    print("Cityscapes val multilabel already exists.")

print("\nAll multilabel files ready!")

## Step 4: Train or Load DAMP Full Checkpoint

`configs/trainers/damp_synthia_fast.yaml` is now the source of truth for the debugged run:

- 3 epochs for `prompt_learner` and `context_decoder`
- CLIP pixel normalization, not ImageNet normalization
- `TRAINER.DAMP.TAU = 0.52`
- `TRAINER.DAMP.PSEUDO_TEMP = 0.0`, meaning logits are calibrated by CLIP logit scale before sigmoid
- checkpoint every epoch

The generated `prompt_learner.pth` includes both `prompt_learner` and `context_decoder`, so CAM generation without `--damp_disable_decoder` is DAMP full.


In [ ]:
# ===== CELL 4: TRAIN / LOAD DAMP FULL =====
%cd {REPO_DIR}

PROMPT_CKPT = DAMP_DIR / 'prompt_learner.pth'

if PROMPT_CKPT.exists():
    print(f'DAMP checkpoint already exists: {PROMPT_CKPT}')
    print('Delete the run directory if you want to retrain from scratch.')
else:
    print(f'Training DAMP full checkpoint -> {DAMP_DIR}')
    !python train.py \
        --config-file configs/trainers/damp_synthia_fast.yaml \
        DATASET.ROOT {DATA_ROOT} \
        OUTPUT_DIR {DAMP_DIR}

# Verify.
if PROMPT_CKPT.exists():
    import os
    size_mb = os.path.getsize(PROMPT_CKPT) / 1024 / 1024
    print(f'\nPrompt checkpoint: {PROMPT_CKPT} ({size_mb:.1f} MB)')
else:
    raise FileNotFoundError(f'prompt_learner.pth not found: {PROMPT_CKPT}')


## Step 5: Generate Zero-shot and DAMP Prompt-only CAMs for 1000 Images

This is the final resource-saving comparison:

- `zero`: CLIP-ES zero-shot CAMs.
- `prompt`: learned DAMP prompt, with `context_decoder` disabled.

We skip DAMP full here because the 3000-image run collapsed around `0.08`.


In [ ]:
# ===== CELL 5: GENERATE ZERO-SHOT + DAMP PROMPT-ONLY CAMs (1000 images) =====
%cd {REPO_DIR}

import glob
from pathlib import Path

# Create an explicit 1000-image split so CAMs, metrics, CRF masks, and train pairs match exactly.
SYNTHIA_CAM_SPLIT.parent.mkdir(parents=True, exist_ok=True)
with open(SYNTHIA_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]
entries_subset = entries[:CAM_MAX_IMAGES]
with open(SYNTHIA_CAM_SPLIT, 'w') as f:
    f.write('\n'.join(entries_subset) + '\n')
print(f'Wrote CAM split: {SYNTHIA_CAM_SPLIT} ({len(entries_subset)} entries)')

for d in (CAM_ZERO_DIR, CAM_PROMPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
if n_zero >= CAM_MAX_IMAGES:
    print(f'Zero-shot CAMs already exist ({n_zero}).')
else:
    print(f'Generating zero-shot CAMs -> {CAM_ZERO_DIR}')
    !python generate_cams.py \
        --dataset synthia \
        --img_root {SYNTHIA_IMG} \
        --label_root {SYNTHIA_LBL} \
        --split_file {SYNTHIA_CAM_SPLIT} \
        --cam_out_dir {CAM_ZERO_DIR} \
        --cam_score softmax \
        --max_images {CAM_MAX_IMAGES} \
        --max_long_side 1024 \
        --num_workers 1 \
        --skip_existing \
        --no_refine

n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
if n_prompt >= CAM_MAX_IMAGES:
    print(f'DAMP prompt-only CAMs already exist ({n_prompt}).')
else:
    print(f'Generating DAMP prompt-only CAMs -> {CAM_PROMPT_DIR}')
    !python generate_cams.py \
        --dataset synthia \
        --img_root {SYNTHIA_IMG} \
        --label_root {SYNTHIA_LBL} \
        --split_file {SYNTHIA_CAM_SPLIT} \
        --cam_out_dir {CAM_PROMPT_DIR} \
        --damp_prompt_ckpt {PROMPT_CKPT} \
        --damp_name_mode train \
        --damp_disable_decoder \
        --cam_score raw \
        --max_images {CAM_MAX_IMAGES} \
        --max_long_side 1024 \
        --num_workers 1 \
        --skip_existing \
        --no_refine

n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
print(f'\nzero CAMs  : {n_zero} in {CAM_ZERO_DIR}')
print(f'prompt CAMs: {n_prompt} in {CAM_PROMPT_DIR}')
if n_zero == 0 or n_prompt == 0:
    raise RuntimeError('Missing CAM files. Check generation logs above.')


## Step 6: Build Class-wise Hybrid CAMs

Hybrid is the last low-resource trick: use the source that was stronger by class in the 20-image ablation.

- zero-shot: `road`, `building`, `wall`, `pole`, `car`, `bicycle`
- DAMP prompt-only: `sidewalk`, `fence`, `vegetation`, `person`, `rider`, `bus`
- other classes fall back to zero-shot unless missing.


In [ ]:
# ===== CELL 6: MERGE ZERO-SHOT + PROMPT-ONLY INTO HYBRID CAMs =====
import glob
import numpy as np
from pathlib import Path
from tqdm import tqdm

CAM_TYPE = 'attn_highres'
CAM_HYBRID_DIR.mkdir(parents=True, exist_ok=True)

# Cityscapes train IDs: 0 road, 1 sidewalk, 2 building, 3 wall, 4 fence,
# 5 pole, 8 vegetation, 11 person, 12 rider, 13 car, 15 bus, 18 bicycle.
PROMPT_CLASSES = {1, 4, 8, 11, 12, 15}
ZERO_CLASSES = {0, 2, 3, 5, 13, 18}

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

saved = 0
missing = []
for entry in tqdm(entries, desc='hybrid'):
    stem = Path(entry).stem
    out_path = CAM_HYBRID_DIR / f'{stem}.npy'
    if out_path.exists():
        saved += 1
        continue

    zero_path = CAM_ZERO_DIR / f'{stem}.npy'
    prompt_path = CAM_PROMPT_DIR / f'{stem}.npy'
    if not zero_path.exists() or not prompt_path.exists():
        missing.append(stem)
        continue

    zero = np.load(zero_path, allow_pickle=True).item()
    prompt = np.load(prompt_path, allow_pickle=True).item()
    zero_keys = [int(x) for x in zero['keys'].tolist()]
    prompt_keys = [int(x) for x in prompt['keys'].tolist()]
    zero_map = {k: i for i, k in enumerate(zero_keys)}
    prompt_map = {k: i for i, k in enumerate(prompt_keys)}
    keys = sorted(set(zero_keys) | set(prompt_keys))

    merged = []
    for k in keys:
        use_prompt = k in PROMPT_CLASSES
        if use_prompt and k in prompt_map:
            merged.append(prompt[CAM_TYPE][prompt_map[k]])
        elif (not use_prompt) and k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])
        elif k in prompt_map:
            merged.append(prompt[CAM_TYPE][prompt_map[k]])
        elif k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])

    if not merged:
        missing.append(stem)
        continue

    out = dict(prompt)
    out[CAM_TYPE] = np.stack(merged, axis=0).astype(prompt[CAM_TYPE].dtype, copy=False)
    out['keys'] = np.asarray(keys, dtype=np.int64)
    np.save(out_path, out)
    saved += 1

n_hybrid = len(glob.glob(str(CAM_HYBRID_DIR / '*.npy')))
print(f'Hybrid CAMs: {n_hybrid} in {CAM_HYBRID_DIR}')
if missing:
    print(f'WARNING: missing {len(missing)} hybrid entries. First 10: {missing[:10]}')
if n_hybrid == 0:
    raise RuntimeError('No hybrid CAMs generated.')


## Step 7: Evaluate All CAM Sets on the Same 1000-image Split

This gives the final decision table. Use the best row by mIoU as `BEST_CAM_KIND` before running CRF.


In [ ]:
# ===== CELL 7: EVALUATE ZERO / PROMPT / HYBRID ON 1000 IMAGES =====
%cd {REPO_DIR}

import glob

for kind, cam_dir in CAM_DIR_BY_KIND.items():
    n_cam = len(glob.glob(str(cam_dir / '*.npy')))
    print('\n' + '=' * 80)
    print(f'Evaluating {kind}: {cam_dir}')
    print(f'CAM files found: {n_cam}')
    if n_cam < EVAL_MAX_IMAGES:
        print(f'WARNING: only {n_cam} CAMs exist but EVAL_MAX_IMAGES={EVAL_MAX_IMAGES}.')

    !python eval_cam.py \
        --dataset synthia \
        --cam_out_dir {cam_dir} \
        --gt_root {SYNTHIA_LBL} \
        --split_file {SYNTHIA_CAM_SPLIT} \
        --cam_type attn_highres \
        --max_images {EVAL_MAX_IMAGES} \
        --thres_start 0.005 \
        --thres_end 0.20 \
        --thres_step 0.005

print('\nDecision: set BEST_CAM_KIND in config to the best of zero/prompt/hybrid, then run Step 8.')
print(f'Current BEST_CAM_KIND={BEST_CAM_KIND}, BEST_CAM_DIR={BEST_CAM_DIR}')


## Step 8: Create Pseudo Masks from the Selected Best CAM Set

Default mode is fast CAM argmax/threshold mask generation. Full DenseCRF is optional because it is very slow on 1000 high-resolution SYNTHIA images and can look frozen in Colab.

Set `USE_CRF = True` in the config only if you have enough time/resource.


In [ ]:
# ===== CELL 8: SELECTED CAMs -> PSEUDO-MASKS =====
%cd {REPO_DIR}

import glob
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from cam.evaluate import entry_stem, load_pred_from_npy

BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]
MASK_DIR = OUTPUT_DIR / 'synthia' / f'pseudo_masks_{BEST_CAM_KIND}_{RUN_NAME}_{CAM_MAX_IMAGES}'
SEG_TRAIN_PAIRS = SEG_EXPORT_DIR / f'train_pairs_{BEST_CAM_KIND}_first{CAM_MAX_IMAGES}.txt'

print(f'BEST_CAM_KIND        : {BEST_CAM_KIND}')
print(f'BEST_CAM_DIR         : {BEST_CAM_DIR}')
print(f'MASK_DIR             : {MASK_DIR}')
print(f'USE_CRF              : {USE_CRF}')
print(f'PSEUDO_MASK_THRESHOLD: {PSEUDO_MASK_THRESHOLD}')

MASK_DIR.mkdir(parents=True, exist_ok=True)
n_existing_masks = len(glob.glob(str(MASK_DIR / '*.png')))

if n_existing_masks >= CAM_MAX_IMAGES:
    print(f'Pseudo masks already generated ({n_existing_masks}). Skipping mask generation.')
elif USE_CRF:
    print('Running DenseCRF. This can be very slow for 1000 high-res images.')
    !python eval_cam.py \
        --dataset synthia \
        --cam_out_dir {BEST_CAM_DIR} \
        --gt_root {SYNTHIA_LBL} \
        --split_file {SYNTHIA_CAM_SPLIT} \
        --cam_type attn_highres \
        --max_images {CAM_MAX_IMAGES} \
        --crf \
        --image_root {SYNTHIA_IMG} \
        --mask_output_dir {MASK_DIR} \
        --crf_confidence {CRF_CONFIDENCE} \
        --crf_n_jobs {CRF_N_JOBS}
else:
    print('Fast pseudo-mask generation from CAM argmax + background threshold.')
    with open(SYNTHIA_CAM_SPLIT, 'r') as f:
        entries = [line.strip() for line in f if line.strip()][:CAM_MAX_IMAGES]

    saved = 0
    missing = []
    for entry in tqdm(entries, desc='pseudo masks'):
        stem = entry_stem(entry)
        cam_path = BEST_CAM_DIR / f'{stem}.npy'
        out_path = MASK_DIR / f'{stem}.png'
        if out_path.exists():
            saved += 1
            continue
        if not cam_path.exists():
            missing.append(stem)
            continue

        pred = load_pred_from_npy(
            str(cam_path),
            cam_type='attn_highres',
            cam_eval_thres=float(PSEUDO_MASK_THRESHOLD),
            use_bg_channel=True,
            n_class=19,
            dataset='synthia',
        )
        Image.fromarray(pred.astype(np.uint8)).save(out_path)
        saved += 1

    if missing:
        print(f'WARNING: missing {len(missing)} CAM files. First 10: {missing[:10]}')
    print(f'Saved/kept {saved} pseudo masks.')

n_masks = len(glob.glob(str(MASK_DIR / '*.png')))
print(f'\n{n_masks} pseudo masks saved to {MASK_DIR}')
if n_masks == 0:
    raise RuntimeError('No pseudo masks generated. Check CAM dir and threshold above.')


## Step 9: Export Segmentation Training Pairs

This writes `image_path mask_path` pairs for the selected pseudo-mask set.


In [ ]:
# ===== CELL 9: EXPORT IMAGE/MASK PAIRS FOR SEGMENTATION TRAINING =====
from pathlib import Path

SEG_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

pairs = []
missing = []
for entry in entries:
    stem = Path(entry).stem
    image_path = SYNTHIA_IMG / f'{stem}.png'
    if not image_path.exists():
        image_path = SYNTHIA_IMG / Path(entry).name
    mask_path = MASK_DIR / f'{stem}.png'

    if image_path.exists() and mask_path.exists():
        pairs.append((image_path, mask_path))
    else:
        missing.append((image_path, mask_path))

with open(SEG_TRAIN_PAIRS, 'w') as f:
    for image_path, mask_path in pairs:
        f.write(f'{image_path} {mask_path}\n')

print(f'Exported {len(pairs)} image/mask pairs -> {SEG_TRAIN_PAIRS}')
if missing:
    print(f'WARNING: {len(missing)} entries missing image or mask. First 5:')
    for image_path, mask_path in missing[:5]:
        print(' ', image_path, '|', mask_path)

print('\nSegmentation training inputs:')
print(f'  image_dir : {SYNTHIA_IMG}')
print(f'  mask_dir  : {MASK_DIR}')
print(f'  pair_file : {SEG_TRAIN_PAIRS}')
print(f'  ignore_id : 255')
print(f'  classes   : Cityscapes train IDs, SYNTHIA-valid subset')
